# Without labels — and the branches not taken

> Clustering, PCA and autoencoders in practice, then an honest tour of everything this path skipped and what it would cost you to pick it up.

Read this chapter at `/learn/15-unsupervised-and-the-rest/`. Exported from `src/content/chapters/15-unsupervised-and-the-rest.mdx` — edit there, not here.


Every model so far has been told the right answer.

Today: what you can do when nobody will tell you — and then a deliberately quick
tour of the entire field this fortnight has driven straight past, so that none of
the names are strangers.

## Clustering

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_blobs

X, true_labels = make_blobs(n_samples=400, centers=4, cluster_std=1.1, random_state=7)

def kmeans(X, k, iters=25, seed=0):
    rng = np.random.default_rng(seed)
    centres = X[rng.choice(len(X), k, replace=False)]      # start on real points
    for _ in range(iters):
        d = ((X[:, None, :] - centres[None, :, :]) ** 2).sum(-1)   # (n, k) distances
        assign = d.argmin(1)                                        # nearest centre
        for j in range(k):                                          # move each centre
            if (assign == j).any():
                centres[j] = X[assign == j].mean(0)
    return assign, centres

assign, centres = kmeans(X, 4)
print("cluster sizes:", np.bincount(assign))

Two steps, alternated until nothing moves:

1. Assign each point to its nearest centre.
2. Move each centre to the mean of its points.

That's the entire algorithm. And the whole idea is visible in that one distance
line — broadcasting an `(n, 1, 2)` against a
`(1, k, 2)` to get every point-to-centre distance at once, no loops.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.2))
ax[0].scatter(X[:, 0], X[:, 1], c=true_labels, s=10, cmap="tab10")
ax[0].set_title("the truth (which you would not have)")
ax[1].scatter(X[:, 0], X[:, 1], c=assign, s=10, cmap="tab10")
ax[1].scatter(centres[:, 0], centres[:, 1], c="k", marker="X", s=120)
ax[1].set_title("what k-means found")
for a in ax: a.set_xticks([]); a.set_yticks([])
plt.tight_layout()

Three things will bite you here, and all three are consequences of the algorithm
rather than bugs in it.

**You have to choose k.** There is no principled answer. The elbow method and
silhouette score are heuristics, not criteria, and anyone who tells you otherwise
is selling something.

**It assumes round, similar-sized clusters**, because it measures Euclidean
distance to a mean. Elongated or nested shapes defeat it completely — DBSCAN and
spectral clustering exist for exactly that reason.

**There is no validation set.** And this is the deep discomfort of all
unsupervised work: nothing tells you the answer is right. Nothing can.

The only real test is whether the clusters are *useful to whoever asked*, which
is a conversation, not a metric. Go into that conversation knowing it's coming.

In [ ]:
inertias = []
for k in range(1, 9):
    a, c = kmeans(X, k)
    inertias.append(((X - c[a]) ** 2).sum())
plt.figure(figsize=(4.8, 2.8))
plt.plot(range(1, 9), inertias, "o-"); plt.axvline(4, ls=":", c="crimson")
plt.xlabel("k"); plt.ylabel("within-cluster sum of squares"); plt.tight_layout()
print("inertia always falls with k — at k = n it is exactly zero, and useless")

Read that printed line carefully, because it's the same trap as training accuracy
in chapter 6. The metric improves monotonically as you add capacity, so you can't
choose k by minimising it. At k = n every point is its own cluster and the
inertia is exactly zero, which is perfect and tells you nothing.

## Dimensionality reduction

In [ ]:
from sklearn.datasets import load_digits
d = load_digits()
Xd = d.data - d.data.mean(0)                  # centre first; PCA requires it

# PCA via SVD — the numerically sane way
U, S, Vt = np.linalg.svd(Xd, full_matrices=False)
explained = S ** 2 / (S ** 2).sum()

print(f"64 original dimensions")
for k in [2, 8, 16, 32]:
    print(f"  first {k:2d} components explain {explained[:k].sum():.1%} of the variance")

PCA finds the directions of greatest variance and re-expresses the data in them.

I like framing it this way: **nothing is discarded until you truncate.** PCA
itself is a rotation — you could rotate back and recover the data exactly. The
compression happens entirely in the decision to keep only the first k directions.

In [ ]:
proj = Xd @ Vt[:2].T
plt.figure(figsize=(5.4, 4))
sc = plt.scatter(proj[:, 0], proj[:, 1], c=d.target, s=7, cmap="tab10")
plt.colorbar(sc, label="true digit"); plt.xticks([]); plt.yticks([])
plt.title("digits projected onto their two principal components")
plt.tight_layout()

The digits partly separate — from two numbers each, with no labels used anywhere.

And to be clear about what you're looking at: **PCA never saw the colours.** They
were painted on afterwards, so you could check what it recovered. The structure
was in the pixels the whole time.

**UMAP** and **t-SNE** produce much prettier separations, and they're what you'll
see in papers.

They're non-linear and optimise for preserving *local* neighbourhoods, so clusters
come out crisp and well-separated. Very satisfying to look at.

Here's the warning that belongs beside every one of those plots, though. In a UMAP
embedding, **distances between clusters are not meaningful**. Cluster sizes are
not meaningful. And the whole layout changes with the random seed.

They're excellent for answering "are there groups here at all?" and actively
misleading for "how different are these two groups?" — which is, unfortunately,
the question people usually ask them.

PCA is less pretty and its axes mean something. Choose accordingly.

## Autoencoders

Train a network to reproduce its own input — through a narrow bottleneck. The
bottleneck has no choice but to become a compressed description.

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
import torch, torch.nn as nn

Xt = torch.tensor(d.data / 16.0, dtype=torch.float32)

class AutoEncoder(nn.Module):
    def __init__(self, n_in=64, latent=8):
        super().__init__()
        self.encode = nn.Sequential(nn.Linear(n_in, 32), nn.ReLU(), nn.Linear(32, latent))
        self.decode = nn.Sequential(nn.Linear(latent, 32), nn.ReLU(), nn.Linear(32, n_in))
    def forward(self, x):
        return self.decode(self.encode(x))

torch.manual_seed(0)
ae = AutoEncoder()
opt = torch.optim.AdamW(ae.parameters(), lr=3e-3)
for epoch in range(400):
    opt.zero_grad()
    loss = nn.functional.mse_loss(ae(Xt), Xt)     # the target IS the input
    loss.backward(); opt.step()
print(f"reconstruction MSE: {loss.item():.4f}   (8 numbers stand in for 64)")

In [ ]:
# needs PyTorch (this kernel has it; the browser runtime does not)
with torch.no_grad():
    recon = ae(Xt[:8]).reshape(-1, 8, 8).numpy()
fig, ax = plt.subplots(2, 8, figsize=(9, 2.4))
for i in range(8):
    ax[0, i].imshow(d.images[i], cmap="gray"); ax[0, i].axis("off")
    ax[1, i].imshow(recon[i], cmap="gray"); ax[1, i].axis("off")
ax[0, 0].set_title("original", fontsize=8, loc="left")
ax[1, 0].set_title("through 8 numbers", fontsize=8, loc="left")
plt.tight_layout()

Look at that loss line again: `mse_loss(ae(Xt), Xt)`.

**The label is the input.** This is self-supervision again — precisely the same
trick as masked language modelling, wearing different clothes. Hide something,
predict it, get free labels. Third time we've met it.

And autoencoders hand you **anomaly detection** for free: train on normal data,
and anything that reconstructs badly is unlike what you trained on. That's a
genuinely useful production pattern, and it needs no labelled anomalies — which
is exactly the situation you're usually in, because if you had labelled anomalies
you'd have trained a classifier.

**"Why do I have to centre the data before PCA?"** Because PCA finds directions of
maximum *variance*, and variance is measured about the mean. Skip the centring
and your first component is mostly "the average digit", which is not a direction
of variation at all.

**"What's the difference between PCA and an autoencoder?"** An autoencoder with
linear layers and no activation *is* PCA, essentially. The nonlinearity is what
buys you a curved manifold instead of a flat subspace.

**"My k-means gives different answers every run."** Correct, and exercise 1 is
about exactly that. Set the seed, or run it ten times and keep the best inertia —
which is what `n_init` does in scikit-learn.

**"How do I know how many clusters there really are?"** You don't. This is not
you missing a technique — it is a genuine property of the problem. Bring domain
knowledge, or bring a human, or reframe it as something supervised.

## The branches this path did not take

Everything below is real, used in production somewhere today, and deliberately
out of scope for a fortnight.

This section exists so the names aren't strangers — because knowing **where a
thing lives** is most of what it takes to pick it up later. Each of these has a
fuller treatment in the [extras](/extras/) if it catches your eye.

### Generative models

**GANs** (2014) pit a generator against a discriminator: one makes fakes, the
other tries to spot them, and both improve by competing. They produced the first
genuinely convincing synthetic faces and were the story of the late 2010s.

They're also notoriously unstable to train. *Mode collapse* — where the generator
finds one good output and stops exploring — is a permanent hazard. Largely
superseded for images now.

**Diffusion models** (practical from 2020) learn to remove noise, one small step
at a time, from pure static to an image. Training is stable and the objective is
a simple regression, which is exactly why they beat GANs. Everything you've seen
from image, video and audio generators is this.

Generating a photorealistic image is an extremely hard problem. Predicting noise
is an easy one.

So here's the move. Take a real image. Add a known amount of Gaussian noise to
it. Train a network to predict *the noise you just added*.

That's an ordinary supervised regression problem, with perfect free labels,
because you generated the noise yourself and know exactly what it was.

Then, to generate something new: start from pure noise, and repeatedly subtract
the noise the network predicts. Step by step, static resolves into an image.

A hard generative problem was converted into an easy predictive one — and the
conversion cost nothing, because the labels were free by construction.

That's a move worth being able to recognise, because it's the same shape as
self-supervision in chapter 3 and the autoencoder above. **When a problem is
hard, look for a related prediction problem whose answers you already have.** It
comes up again and again, and it's one of the genuinely transferable ideas in
this field.

### Reinforcement learning

Learning from delayed, evaluative reward rather than from correct answers. The
agent's own actions determine what data it sees next — which is precisely what
makes it genuinely harder than everything else in this book.

Vocabulary you'll meet: *state*, *action*, *policy*, *value function*,
*Q-learning*, *policy gradient*, *PPO*. It's how game-playing and robot control
work, and — as RLHF — how a base language model becomes an assistant.

Budget a month, not an afternoon. I'd rather tell you that than have you bounce
off it.

### Graph neural networks

Message passing over the edges of a graph, so each node's representation is built
from its neighbours'. The right tool when your data genuinely *is* a graph:
molecules, social networks, fraud rings, road systems.

And notice the pattern from [Chapter 11](/learn/11-vision-and-transfer/) — it's
the same "share weights over a structure" idea as convolution, with a graph in
place of a grid. Once you've seen that idea twice, you start seeing it
everywhere. Attention is arguably a third instance, on a fully-connected graph.

### The probabilistic tradition

Naive Bayes, hidden Markov models, Gaussian processes, variational inference.

Its selling point is **calibrated uncertainty**: not just a prediction, but an
honest statement of how sure the model is. Which is worth a great deal in some
settings and nothing at all in others.

It lost the scaling race, and it's very much alive wherever data is scarce or
being wrong is expensive — clinical trials, scientific experiment design,
small-sample forecasting. Bayes' rule is the entry point,
and it's a short walk from what you already know.

### Classical time series

ARIMA, exponential smoothing, Prophet, state-space models.

Frequently better than a neural network on a single series with a few hundred
points, and the honest default for forecasting problems. Do not let anybody
embarrass you out of using a 1970s method that works.

The critical discipline is the one from
[Chapter 6](/learn/06-generalisation/): validate *forward in time*, never
randomly. You already have that reflex.

The reason all of these fit in one section rather than one chapter each is not
that they're unimportant.

It's that they share the machinery you have already built — a model, a loss,
gradients, a validation strategy — and differ mainly in what they *assume about
the data*.

Which is the real payoff of the last fortnight. You now have the frame that all
of them slot into. So picking any one of them up is learning a specific set of
assumptions, rather than learning a field from scratch.

That's the difference between a fortnight and a career, and you've just done the
fortnight.

In [ ]:
# 1. Run kmeans with k=4 from five different seeds. Do you always get
#    the same clustering? Compare inertias.
#
# 2. Reconstruct the digits from only their first k principal components,
#    for k = 2, 8, 16, 32. At what point are they recognisable?
#
# 3. Take one digit class (say 3), fit PCA on the other nine, and measure
#    reconstruction error for both groups. You have just built an
#    anomaly detector with no anomaly labels.

print("replace me")

For 3, the logic is: fit a compression to what "normal" looks like, then measure
how badly each thing compresses. Things unlike the training data should compress
badly.

Predict roughly how well you think it'll work before you run it — and then read
the numbers honestly. That's the actual exercise.

In [ ]:
print("k-means is sensitive to initialisation:")
for seed in range(5):
    a, c = kmeans(X, 4, seed=seed)
    print(f"  seed {seed}: inertia {((X - c[a]) ** 2).sum():9.1f}   sizes {np.bincount(a, minlength=4)}")

Different seeds, different answers — sometimes materially different.

k-means finds a **local** optimum, not the global one. Real implementations run it
ten times from different starts and keep the best inertia (`n_init=10` in
scikit-learn), and use k-means++ initialisation, which deliberately spreads the
initial centres apart. Neither trick guarantees anything; both help a lot.

In [ ]:
fig, ax = plt.subplots(5, 6, figsize=(7.5, 6.2))
for row, k in enumerate([2, 4, 8, 16, 64]):
    approx = (Xd @ Vt[:k].T) @ Vt[:k] + d.data.mean(0)
    for col in range(6):
        ax[row, col].imshow(approx[col].reshape(8, 8), cmap="gray"); ax[row, col].axis("off")
    ax[row, 0].set_title(f"k={k}", fontsize=8, loc="left")
plt.tight_layout()

Around 8–16 components the digits become clearly readable. That's a compression
from 64 numbers down to 16 — and it's the same trade the autoencoder made, with a
linear model instead of a network.

In [ ]:
normal = d.data[d.target != 3]
odd    = d.data[d.target == 3]
mu = normal.mean(0)
_, _, V = np.linalg.svd(normal - mu, full_matrices=False)
k = 8

def recon_error(A):
    C = A - mu
    return (((C @ V[:k].T) @ V[:k] - C) ** 2).mean(1)

en, eo = recon_error(normal), recon_error(odd)
print(f"reconstruction error, digits 0-9 except 3 : {en.mean():.2f}")
print(f"reconstruction error, digit 3 (unseen)    : {eo.mean():.2f}")
thresh = np.percentile(en, 95)
far, det = (en > thresh).mean(), (eo > thresh).mean()
print(f"\nflagging error > {thresh:.1f} (95th pct of normal):")
print(f"  false alarm rate on normal : {far:.1%}")
print(f"  detection rate on digit 3  : {det:.1%}")
print(f"  lift over chance           : {det / far:.1f}x")

You've just built an anomaly detector out of data containing no anomaly labels.
The principle generalises directly: **fit a compression to what normal looks
like, then flag whatever refuses to compress.**

Now — read those numbers honestly, because this is the important part.

It catches about 14% of the unseen digit at a 5% false-alarm rate. That's nearly
three times better than chance, and it is a **weak** detector. I could have picked
a friendlier example and shown you a bigger number, and I'd rather not.

That's not a failure of the code. It's the expected result when the "anomaly"
looks a great deal like the normal data. A handwritten 3 shares most of its
strokes with an 8, a 5 and a 9 — so a subspace fitted to those reconstructs it
fairly well. The detector is telling you something true about the world.

Two things worth carrying away.

**Unsupervised anomaly detection is a screening tool.** It narrows a million rows
down to a thousand for a human to look at. It is not a decision procedure, and
promising one is how these projects get people into trouble.

**And the threshold is a choice.** Move it and you trade false alarms against
missed detections. Where you put it depends entirely on what each of those costs
you — which is a question about your organisation, not about your model.

That last sentence is tomorrow's chapter. It's the point where machine learning
stops being a technical activity.

Tomorrow, the last day: turning a model into something someone can rely on, and
learning to read the literature on your own.